# Analisis Sentimen: Performa Timnas Indonesia

In [ ]:
# Install pustaka yang dibutuhkan jika menjalankan di lingkungan baru
!pip install -q pandas matplotlib seaborn scikit-learn transformers torch

## 0. Konfigurasi Proyek & Parameter

In [ ]:
import os
import re
import glob
import json
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- KONFIGURASI UTAMA ---
POLA_FILE = 'dataset_timnas.csv'       
KEYWORD = 'Timnas Indonesia'
METODE = 'indoroberta'                 
JUMLAH_SAMPEL = 20                     
SEED = 42                              
WAJIB_LABEL_DULU = True                

# Path Output
FOLDER_OUTPUT = 'output'
FILE_EVALUASI = 'evaluasi_sentimen.csv' 
FILE_HASIL_SENTIMEN = os.path.join(FOLDER_OUTPUT, 'hasil_sentimen.csv')
FILE_RINGKASAN = os.path.join(FOLDER_OUTPUT, 'ringkasan_sentimen.csv')
FILE_GRAFIK = os.path.join(FOLDER_OUTPUT, 'distribusi_sentimen.png')

# Pemetaan label untuk konsistensi
URUTAN_SENTIMEN = ['positive', 'neutral', 'negative']
PETA_LABEL = {
    'positive': 'positive', 'positif': 'positive', 'pos': 'positive',
    'neutral': 'neutral', 'netral': 'neutral', 'net': 'neutral',
    'negative': 'negative', 'negatif': 'negative', 'neg': 'negative'
}

os.makedirs(FOLDER_OUTPUT, exist_ok=True)
print('Konfigurasi berhasil dimuat. Folder output siap.')

## 1. Pembuatan & Pembersihan Data (Data Preprocessing)

In [ ]:
# Load dataset
if os.path.exists(POLA_FILE):
    df = pd.read_csv(POLA_FILE)
else:
    files = glob.glob(POLA_FILE)
    assert len(files) > 0, f'File dataset {POLA_FILE} tidak ditemukan!'
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

print(f'Total baris awal: {len(df)}')

# Deteksi nama kolom teks (dukung 'text' atau 'full_text')
if 'text' in df.columns:
    KOLOM_TEKS = 'text'
elif 'full_text' in df.columns:
    KOLOM_TEKS = 'full_text'
    df['text'] = df['full_text']
else:
    raise KeyError("Dataset harus memiliki kolom 'text' atau 'full_text'.")

# Pastikan kolom wajib ada
for col in ['created_at', 'like_count']:
    if col not in df.columns:
        if col == 'like_count' and 'favorite_count' in df.columns:
            df['like_count'] = df['favorite_count']
        else:
            df[col] = 0 if col == 'like_count' else datetime.now().strftime('%Y-%m-%d %H:%M:%S')

# Hapus duplikat & missing values pada kolom teks
awal = len(df)
df = df.dropna(subset=['text']).drop_duplicates(subset=['text']).reset_index(drop=True)
df['id_post'] = df.index
print(f'Setelah pembersihan duplikat & nilai kosong: {len(df)} postingan (Dibuang: {awal - len(df)})')

In [ ]:
def bersihkan_teks_sentimen(teks):
    '''
    Pembersihan teks terukur untuk Analisis Sentimen:
    - Mengubah huruf menjadi kecil (lowercase)
    - Menghapus URL, Mention (@username), dan simbol berlebih
    - PERTAHANKAN kata negasi (tidak, bukan, gak, kurang, jangan, belum)
    '''
    s = str(teks).lower()
    s = re.sub(r'http\S+|www\.\S+', '', s)       # Hapus URL
    s = re.sub(r'@\w+', '', s)                   # Hapus Mention
    s = re.sub(r'\brt\b', '', s)                 # Hapus kata RT
    s = re.sub(r'#(\w+)', r'\1', s)               # Ambil kata dari hashtag
    s = re.sub(r'[^a-z0-9\s]', ' ', s)           # Hapus karakter khusus
    s = re.sub(r'\s+', ' ', s).strip()           # Rapikan spasi
    return s

# Buat kolom cleaned_text
df['cleaned_text'] = df['text'].apply(bersihkan_teks_sentimen)

# Buang postingan yang menjadi kosong setelah pembersihan
df = df[df['cleaned_text'] != ''].reset_index(drop=True)
df['id_post'] = df.index

print(f'Jumlah data final siap dianalisis: {len(df)}')
df[['id_post', 'text', 'cleaned_text']].head(3)

## 2. Pengambilan 20 Postingan Acak (Sebelum Klasifikasi Sistem)

In [ ]:
# Pengambilan 20 sampel acak
sampel = df.sample(n=JUMLAH_SAMPEL, random_state=SEED)[['id_post', 'text']].copy()
sampel = sampel.rename(columns={'text': 'teks'}).sort_values('id_post')
sampel['label_manual'] = ''
sampel['prediksi'] = ''
sampel['catatan'] = ''

if os.path.exists(FILE_EVALUASI):
    print(f'PERINGATAN: File `{FILE_EVALUASI}` sudah ada. Tidak ditimpa agar label manual aman.')
else:
    sampel.to_csv(FILE_EVALUASI, index=False)
    print(f'Berhasil mengekstrak 20 sampel acak ke file `{FILE_EVALUASI}`.')

# Tampilkan 5 sampel pertama
pd.read_csv(FILE_EVALUASI).head(5)

## 3. Klasifikasi Sentimen Sistem

In [ ]:
def muat_label_manual():
    m = pd.read_csv(FILE_EVALUASI)
    m['label_manual_clean'] = m['label_manual'].fillna('').astype(str).str.strip().str.lower().map(PETA_LABEL)
    return m

if WAJIB_LABEL_DULU:
    df_m = muat_label_manual()
    kosong = df_m[df_m['label_manual_clean'].isna()]
    if len(kosong) > 0:
        raise RuntimeError(
            f'Masih ada {len(kosong)} baris di `{FILE_EVALUASI}` yang belum diisi label manualnya! '
            f'(ID Post: {kosong["id_post"].tolist()}). Isi dulu dengan positive/neutral/negative, '
            'lalu jalankan ulang sel ini.'
        )
    print('✅ Label manual 20 sampel sudah terisi lengkap. Melanjutkan klasifikasi sistem...')

In [ ]:
if METODE == 'indoroberta':
    try:
        from transformers import pipeline
        print('Memuat model IndoRoBERTa Sentiment Classifier...')
        clf = pipeline(
            'sentiment-analysis',
            model='w11wo/indonesian-roberta-base-sentiment-classifier',
            truncation=True, max_length=128
        )
        hasil = clf(df['cleaned_text'].tolist(), batch_size=32)
        df['sentiment_raw'] = [h['label'] for h in hasil]
        df['confidence'] = [round(float(h['score']), 4) for h in hasil]
    except Exception as e:
        print(f'Gagal memuat transformer ({e}), menggunakan fallback Lexicon/Rule-based...')
        METODE = 'rule_based'

if METODE == 'rule_based':
    kata_pos = {'menang', 'juara', 'bangga', 'keren', 'mantap', 'terbaik', 'top', 'semangat', 'unggul', 'garuda', 'gol', 'gacor', 'bagus', 'dukung'}
    kata_neg = {'kalah', 'kecewa', 'buruk', 'jelek', 'parah', 'curang', 'wasit', 'out', 'rugi', 'lemah', 'blunder', 'salah', 'kecewa', 'rugi'}
    
    def prediksi_rule(teks):
        tokens = str(teks).lower().split()
        p = sum(1 for t in tokens if t in kata_pos)
        n = sum(1 for t in tokens if t in kata_neg)
        if p > n: return 'positive', 0.85
        elif n > p: return 'negative', 0.85
        else: return 'neutral', 0.70
        
    res = [prediksi_rule(t) for t in df['cleaned_text']]
    df['sentiment_raw'] = [r[0] for r in res]
    df['confidence'] = [r[1] for r in res]

df['sentiment'] = df['sentiment_raw'].astype(str).str.strip().str.lower().map(PETA_LABEL)
print('✅ Klasifikasi sentimen sistem selesai untuk seluruh dataset.')
df[['id_post', 'text', 'sentiment', 'confidence']].head(5)

## 4. Jumlah, Persentase, dan Visualisasi Distribusi Sentimen

In [ ]:
ringkasan = (
    df['sentiment'].value_counts().reindex(URUTAN_SENTIMEN, fill_value=0)
    .rename_axis('sentimen').reset_index(name='jumlah')
)
ringkasan['persentase'] = (ringkasan['jumlah'] / ringkasan['jumlah'].sum() * 100).round(2)

ringkasan.to_csv(FILE_RINGKASAN, index=False)
print('--- RINGKASAN DISTRIBUSI SENTIMEN ---')
print(ringkasan)

In [ ]:
warna_sentimen = {'positive': '#2e9e5b', 'neutral': '#8a94a6', 'negative': '#d64545'}

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    [s.capitalize() for s in ringkasan['sentimen']], 
    ringkasan['jumlah'],
    color=[warna_sentimen[s] for s in ringkasan['sentimen']],
    edgecolor='black',
    alpha=0.85
)

for bar, j, p in zip(bars, ringkasan['jumlah'], ringkasan['persentase']):
    ax.text(
        bar.get_x() + bar.get_width() / 2, 
        bar.get_height() + (ringkasan['jumlah'].max() * 0.02),
        f'{j}\n({p}%)', 
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

ax.set_title(f'Distribusi Sentimen Publik Terhadap Timnas Indonesia (N={len(df)})', fontsize=12, fontweight='bold', pad=15)
ax.set_xlabel('Kategori Sentimen', fontsize=11)
ax.set_ylabel('Jumlah Postingan (Tweets)', fontsize=11)
ax.set_ylim(0, ringkasan['jumlah'].max() * 1.25)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()

plt.savefig(FILE_GRAFIK, dpi=300)
plt.show()
print(f'Grafik disimpan ke `{FILE_GRAFIK}`')

## 5. Evaluasi: Label Manual vs Prediksi Sistem (20 Sampel)

In [ ]:
df_eval = pd.read_csv(FILE_EVALUASI)
df_eval['label_manual'] = df_eval['label_manual'].astype(str).str.strip().str.lower().map(PETA_LABEL)

pred_map = df.set_index('id_post')['sentiment'].to_dict()
conf_map = df.set_index('id_post')['confidence'].to_dict()

df_eval['prediksi'] = df_eval['id_post'].map(pred_map)
df_eval['confidence'] = df_eval['id_post'].map(conf_map)
df_eval['sesuai'] = df_eval['label_manual'] == df_eval['prediksi']

n_total = len(df_eval)
n_sesuai = int(df_eval['sesuai'].sum())
persen_sesuai = round(n_sesuai / n_total * 100, 2)

print('=== HASIL EVALUASI KESESUAIAN (20 SAMPEL) ===')
print(f'Jumlah postingan sesuai : {n_sesuai} dari {n_total}')
print(f'Persentase Kesesuaian   : {persen_sesuai}%\n')

ct = pd.crosstab(df_eval['label_manual'], df_eval['prediksi'], rownames=['Label Manual'], colnames=['Prediksi Sistem'])
ct = ct.reindex(index=URUTAN_SENTIMEN, columns=URUTAN_SENTIMEN, fill_value=0)
print('Tabel Silang (Crosstab):')
print(ct)

df_eval.to_csv(FILE_EVALUASI, index=False)
print(f'✅ File `{FILE_EVALUASI}` diperbarui dengan prediksi sistem & kolom kesesuaian.')

In [ ]:
mismatches = df_eval[~df_eval['sesuai']][['id_post', 'teks', 'label_manual', 'prediksi', 'confidence', 'catatan']]

print(f'=== ANALISIS KETIDAKSESUAIAN ({len(mismatches)} Kasus Ditemukan) ===')
if len(mismatches) == 0:
    print('Sempurna! Semua prediksi sistem sesuai dengan label manual.')
else:
    for i, row in mismatches.head(3).iterrows():
        print(f'\n📌 Contoh Ketidaksesuaian #{row["id_post"]}:')
        print(f'Teks         : "{row["teks"]}"')
        print(f'Label Manual : {row["label_manual"].upper()}')
        print(f'Prediksi Sys : {row["prediksi"].upper()} (Confidence: {row["confidence"]})')
        print(f'Catatan/Anal : {row["catatan"] if pd.notna(row["catatan"]) and row["catatan"] != "" else "Perlu dianalisis (misal: sarkasme/istilah bola/bahasa gaul)"}')

## 6. Ekspor Berkas Integrasi Streamlit

In [ ]:
kolom_ekspor = ['id_post', 'text', 'cleaned_text', 'created_at', 'like_count', 'sentiment', 'confidence']
for c in ['retweet_count', 'reply_count']:
    if c in df.columns:
        kolom_ekspor.append(c)

df_export = df[kolom_ekspor]
df_export.to_csv(FILE_HASIL_SENTIMEN, index=False)

meta = {
    'topik': 'Performa Timnas Indonesia di Kualifikasi Piala Dunia',
    'metode': METODE,
    'total_postingan': int(len(df)),
    'distribusi': {r.sentimen: {'jumlah': int(r.jumlah), 'persentase': float(r.persentase)} for r in ringkasan.itertuples()},
    'evaluasi_manual': {
        'total_sampel': n_total,
        'jumlah_sesuai': n_sesuai,
        'persen_kesesuaian': persen_sesuai
    },
    'diperbarui': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

with open(os.path.join(FOLDER_OUTPUT, 'meta_sentimen.json'), 'w') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print('====================================================')
print('🎉 SELURUH PROSES ANALISIS SENTIMEN SELESAI!')
print('Berkas berikut siap diserahkan ke penanggung jawab Streamlit:')
print(f' 1. `{FILE_EVALUASI}` (Evaluasi 20 sampel & perbandingan manual-sistem)')
print(f' 2. `{FILE_HASIL_SENTIMEN}` (Dataset lengkap dengan kolom sentiment & confidence)')
print(f' 3. `{FILE_RINGKASAN}` (Tabel hitungan & persentase sentimen)')
print(f' 4. `{FILE_GRAFIK}` (Grafik batang distribusi sentimen)')
print('====================================================')